# Merge: wave-2 IR/severity table (dynamic) + reduced census (static)

Rebuilt version. Instead of merging the full daily corona panel
(`Corona_loc_processed.csv`) with the census, this merges the **wave-2
cross-sectional table** built by `build_wave2_ir_severity_table.py`
(`data/processed/models/wave2_ir_severity_table.csv` - one row per area,
window 2020-06-01 to 2020-10-31, 4 target features: `mean_IR`, `peak_IR`,
`slope_to_peak_IR`, `severity_hosp_per_case`) with the reduced census
features (`Mifkad_2_reduced.csv`, 27 columns).

**Join type fix vs. the previous version of this notebook:** the earlier
version of this notebook documented a left join (with unmatched areas kept
as NaN, flagged via `has_census`) but the code actually ran an inner join,
silently dropping ~27% of areas - a real inconsistency between the
notebook's own documentation and its behavior. This version does what was
originally documented: a **left join**, with an explicit `has_census` flag,
so unmatched areas are visible in the output instead of disappearing.

Output is one row per area with 4 corona-side targets + 27 census features +
`has_census` - the direct input to the wave-2 modeling step.


In [1]:
from pathlib import Path
import pandas as pd

CORONA_PATH = Path("/home/bcrlab/igguest/porat_naama/data/processed/models/wave2_ir_severity_table.csv")
MIFKAD_PATH = Path("/home/bcrlab/igguest/porat_naama/data/processed/mifkad/Mifkad_2_reduced.csv")
OUT_DIR = Path("/home/bcrlab/igguest/porat_naama/data/processed/models")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Running list of (path, description) for every file this notebook saves
output_log = []

def log_output(path, description, description_he):
    output_log.append({"file_path": str(path), "contents": description, "contents_he": description_he})
    print("Saved:", path)


## Load both sides

`df_corona` is already one row per area (built by
`build_wave2_ir_severity_table.py`), so this is a simple key-based merge -
no daily panel, no date column.

In [2]:
df_corona = pd.read_csv(CORONA_PATH, dtype={"City_agas_code": str})
df_mifkad = pd.read_csv(MIFKAD_PATH, dtype={"City_agas_code": str})

assert df_corona["City_agas_code"].is_unique, "wave2 IR/severity table key is not unique"
assert df_mifkad["City_agas_code"].is_unique, "Mifkad key is not unique - merge would explode rows"

print(f"corona (wave-2 IR/severity): {df_corona.shape[0]:,} areas, {df_corona.shape[1]} cols")
print(f"mifkad (reduced):            {df_mifkad.shape[0]:,} areas, {df_mifkad.shape[1]} cols")


corona (wave-2 IR/severity): 1,595 areas, 10 cols
mifkad (reduced):            4,002 areas, 28 cols


## Merge

**Left join** on `City_agas_code`: the wave-2 table is the base (it is
already the deliberately-filtered set of areas with enough data - see
`build_wave2_ir_severity_table.py`), census columns broadcast onto it where
a match exists. Unmatched areas are kept with NaN census values, flagged via
`has_census` rather than silently dropped - this is the documented decision
in `COVID-Project-CLAUDE.md` (geographic-code mismatch between COVID
sub-areas and 2022 census `StatArea`), now actually implemented as such.

In [3]:
df_merged = df_corona.merge(
    df_mifkad,
    on="City_agas_code",
    how="left",
    validate="one_to_one",
)

df_merged["has_census"] = df_merged["age_structure_idx"].notna().astype(int)

n_total = len(df_merged)
n_with_census = df_merged["has_census"].sum()
print(f"Areas in wave-2 table: {n_total:,}")
print(f"Areas with a census match: {n_with_census:,} ({100 * n_with_census / n_total:.1f}%)")
print(f"Areas without a census match (kept, NaN census columns): {n_total - n_with_census:,}")


Areas in wave-2 table: 1,595
Areas with a census match: 1,008 (63.2%)
Areas without a census match (kept, NaN census columns): 587


## Report unmatched keys

Areas present in the wave-2 table with no matching `City_agas_code` in the
reduced census data - kept in `df_merged` (per the left join above), also
listed separately here for inspection.

In [4]:
unmatched = df_merged.loc[df_merged["has_census"] == 0, ["City_agas_code", "town", "is_city_aggregate"]]

unmatched_path = OUT_DIR / "unmatched_keys_wave2.csv"
unmatched.sort_values("City_agas_code").to_csv(unmatched_path, index=False, encoding="utf-8-sig")

log_output(
    unmatched_path,
    "Areas from the wave-2 IR/severity table with no matching City_agas_code in the reduced "
    "census data. These rows ARE included in wave2_model_table.csv (left join) with NaN census "
    "columns and has_census=0 - this file just lists them for inspection.",
    "אזורים מתוך טבלת ה-IR/חומרה של הגל השני שאין להם City_agas_code מתאים בנתוני המפקד "
    "המצומצמים. השורות האלה כן נכללות ב-wave2_model_table.csv (left join) עם עמודות מפקד "
    "ריקות (NaN) ו-has_census=0 - קובץ זה רק מפרט אותן לבדיקה.",
)


Saved: /home/bcrlab/igguest/porat_naama/data/processed/models/unmatched_keys_wave2.csv


## Save the final model table

One row per area: 4 wave-2 corona targets (`mean_IR`, `peak_IR`,
`slope_to_peak_IR`, `severity_hosp_per_case`) + `is_city_aggregate` +
underlying totals (`total_tests_in_window` etc.) + 27 reduced census
features + `has_census`. This is the direct input for the modeling step
(univariate screen / combined regression against the census features).

In [5]:
merged_path = OUT_DIR / "wave2_model_table.csv"
df_merged.to_csv(merged_path, index=False, encoding="utf-8-sig")

log_output(
    merged_path,
    "Wave-2 (2020-06-01 to 2020-10-31) corona targets left-joined with the reduced census "
    "features (Mifkad_2_reduced.csv) on City_agas_code. One row per area. Corona-side columns: "
    "mean_IR, peak_IR, slope_to_peak_IR, severity_hosp_per_case, is_city_aggregate, "
    "total_tests_in_window, total_cases_in_window, total_hospitalized_in_window (all from "
    "build_wave2_ir_severity_table.py). Census columns repeat the 27 reduced features; unmatched "
    "areas keep NaN census values (has_census=0) instead of being dropped.",
    "יעדי הגל השני (1.6.2020-31.10.2020) ממוזגים (left join) עם נתוני המפקד המצומצמים "
    "(Mifkad_2_reduced.csv) לפי City_agas_code. שורה אחת לכל אזור. עמודות צד-הקורונה: "
    "mean_IR, peak_IR, slope_to_peak_IR, severity_hosp_per_case, is_city_aggregate, "
    "total_tests_in_window, total_cases_in_window, total_hospitalized_in_window (מ-"
    "build_wave2_ir_severity_table.py). עמודות המפקד הן 27 הפיצ'רים המצומצמים; אזורים ללא "
    "התאמה נשארים עם NaN בעמודות המפקד (has_census=0) במקום להיזרק.",
)


Saved: /home/bcrlab/igguest/porat_naama/data/processed/models/wave2_model_table.csv


In [6]:
manifest_path = OUT_DIR / "merge_output_files_manifest.csv"
pd.DataFrame(output_log).to_csv(manifest_path, index=False, encoding="utf-8-sig")
print("Saved:", manifest_path)


Saved: /home/bcrlab/igguest/porat_naama/data/processed/models/merge_output_files_manifest.csv


## Restrict to areas with both corona and census data

The left join above is kept for transparency (it's how we counted the
63.2%/36.8% match rate and produced `unmatched_keys_wave2.csv`), but a row
with no census match is useless for testing the census<->covid relationship
- every predictor column on it is NaN. The final modeling table therefore
keeps only areas with **both** a wave-2 corona row and a census match
(`has_census == 1`). `has_census` itself is dropped afterward since it would
be constant (always 1) in the filtered table.

In [7]:
n_before = len(df_merged)
df_model = df_merged[df_merged["has_census"] == 1].drop(columns=["has_census"]).reset_index(drop=True)
n_after = len(df_model)

print(f"Rows before filtering to matched areas: {n_before:,}")
print(f"Rows after keeping only has_census==1:   {n_after:,}")
print(f"Rows dropped (no census match):          {n_before - n_after:,}")

df_model.to_csv(merged_path, index=False, encoding="utf-8-sig")

# Replace the earlier (unfiltered) log entry for this same file with the updated description
output_log[:] = [entry for entry in output_log if entry["file_path"] != str(merged_path)]
log_output(
    merged_path,
    "Wave-2 (2020-06-01 to 2020-10-31) corona targets merged with the reduced census features "
    "(Mifkad_2_reduced.csv) on City_agas_code, RESTRICTED to areas with both a corona-side row "
    "and a census match (has_census==1 from the left join above; unmatched areas are excluded "
    "here, listed separately in unmatched_keys_wave2.csv). One row per area. Corona columns: "
    "mean_IR, peak_IR, slope_to_peak_IR, severity_hosp_per_case, is_city_aggregate, "
    "total_tests_in_window, total_cases_in_window, total_hospitalized_in_window. Plus the 27 "
    "reduced census features.",
    "יעדי הגל השני (1.6.2020-31.10.2020) ממוזגים עם נתוני המפקד המצומצמים "
    "(Mifkad_2_reduced.csv) לפי City_agas_code, מוגבל לאזורים עם גם שורת קורונה וגם התאמת "
    "מפקד (has_census==1 מה-left join למעלה; אזורים ללא התאמה מוצאים מכאן, מפורטים בנפרד "
    "ב-unmatched_keys_wave2.csv). שורה אחת לכל אזור. עמודות קורונה: mean_IR, peak_IR, "
    "slope_to_peak_IR, severity_hosp_per_case, is_city_aggregate, total_tests_in_window, "
    "total_cases_in_window, total_hospitalized_in_window. בתוספת 27 פיצ'רי המפקד המצומצמים.",
)


Rows before filtering to matched areas: 1,595
Rows after keeping only has_census==1:   1,008
Rows dropped (no census match):          587
Saved: /home/bcrlab/igguest/porat_naama/data/processed/models/wave2_model_table.csv


In [8]:
manifest_path = OUT_DIR / "merge_output_files_manifest.csv"
pd.DataFrame(output_log).to_csv(manifest_path, index=False, encoding="utf-8-sig")
print("Saved:", manifest_path)


Saved: /home/bcrlab/igguest/porat_naama/data/processed/models/merge_output_files_manifest.csv
